In [3]:
import argparse
from itertools import product
import multiprocessing
import time
import datetime
import pandas as pd
import pymysql
import requests
from dateutil.relativedelta import relativedelta
from scrapy.selector import Selector
import json
from urllib.parse import urlencode
from urllib import parse

# sentower 榜单数据爬取

In [4]:
ios_category_list = ["7003","7004","7005","7006","7007","7015"]
android_category_list = ["all","game","game_casino","game_casual","game_simulation"] #全部分类、休闲、Games/Board、Games/Card、Games/Casino、Games/Dice、模拟类游戏
country_list = ["BR","IN","ID","US","MY","SG","TH","TW","KH","PH","VN"]
ios_charttype_list = ["topfreeapplications","topgrossingapplications","topfreeipadapplications","topgrossingipadapplications"]
android_charttype_list = ["topselling_free","topgrossing","topselling_new_free"]
# def get_info(sql):
#     url_sql="http://8.210.252.30:5000?sql="+sql
#     response_sql = requests.request("get", url=url_sql)
#     sql_datalist=json.loads(response_sql.text)
#     return sql_datalist
def savedata(table,savedata_list):
    values=savedata_list
    #values=str(savedata_list).replace("'","\"").replace(r"\n","")
    #print(values)
    url_update='http://150.230.206.248:5000/update'
    payload={'table': table,
    'values': values}
    headers = {}
    response = requests.request("post", url=url_update, headers=headers, data=payload)
# class with_mysql(object):
#     def __init__(self):
#         self.config = {
#             'host': 'localhost',
#             'port': 3306,
#             'user': 'root',
#             'password': 'milipwd',
#             'db': 'data_analysis',
#             'charset': 'utf8mb4',
#         }
    
#     def get_info(self,sql):
#         conn = pymysql.connect(**self.config)
#         cursor = conn.cursor()
#         try:
#             cursor.execute(sql)
#         except Exception as e:
#             print(e)
#         results = cursor.fetchall()
#         conn.close()
#         return results
        
#     def save_dict(self, table_name, data_dict):
#         conn = pymysql.connect(**self.config)
#         cursor = conn.cursor()
#         table = table_name
#         keys = ', '.join(data_dict.keys())
#         values = ', '.join(['%s'] * len(data_dict))
#         sql = 'REPLACE INTO {table}({keys}) VALUES ({values})'.format(table=table, keys=keys, values=values)
#         try:
#             if cursor.execute(sql, tuple(data_dict.values())):
#                 print('S')
#                 conn.commit()
#         except Exception as e:
#             print(e)
#             conn.rollback()
#         conn.close()

In [7]:
from_=int((datetime.date.today() - datetime.timedelta(days=1)).strftime('%Y%m%d')) #日期选择


headers = {
    'accept': '*/*',
}
auth_token = 'ST0_zxixGoMC_JvC24JFqtLFzxs'

for co in country_list:
    for ca in android_category_list:
        for ch in android_charttype_list:
            print(co+ca+ch)

            params = {
                'category': ca, #app类型
                'chart_type': ch,#榜单类型
                'country': co, #增加或者减少国家
                'date': from_, #爬取日期
                'auth_token': auth_token  #token
            }
            print(params)
            # 请求app排名，获取appid
            response = requests.get('https://api.sensortower-china.com/v1/android/ranking', params=params, headers=headers)
            category = []
            chart_type = []
            date = []
            appid = []
            rank = []
            data = response.json()
            print(data)
            for i in data['ranking']:
                category.append(data["category"])
                chart_type.append(data["chart_type"])
                date.append(data["date"][0:10])
                appid.append(i)

            for r in range(1,len(data['ranking'])+1):
                rank.append(r)

            APPIDdatafram = pd.DataFrame.from_dict ({"ranklist":rank,"date":date,"appid":appid,"category":category,"chart_type":chart_type})[:50] #获得app排名数据
            APPIDdatafram['appid']=APPIDdatafram['appid'].astype(object)
            app_ids = ",".join(APPIDdatafram['appid'])
            ####爬取sentower app详细指标部分
            headers2 = {
                'accept': 'application/json',
            }

            params2 = {
                'app_ids': app_ids,
                'country': co,
                'auth_token': "ST0_zxixGoMC_JvC24JFqtLFzxs"
            }
            response2 = requests.get('https://api.sensortower-china.com/v1/android/apps', params=params2, headers=headers2)
            appinfodata = response2.json()
            ##### sentower指标
            app_id = []
            canonical_country = []
            humanized_name = []
            publisher_name = []
            icon_url = []
            os =[]
            active = []
            url = []
            valid_countries = []
            top_countries = []
            release_date = []
            updated_date = []
            in_app_purchases = []
            rating =[]
            price = []
            rating_count = []
            version = []
            humanized_worldwide_last_month_downloads = []
            humanized_worldwide_last_month_revenue = []
            feature_graphic = []
            short_description = []
            screenshot_urls = []
            categories = []
            
            for a in appinfodata['apps']:
                app_id.append(str(a['app_id']))
                categories.append(str(a['categories']))
                canonical_country.append(str(a['canonical_country']))
                humanized_name.append(str(a['humanized_name']))
                publisher_name.append(str(a['publisher_name']))
                icon_url.append(str(a['icon_url']))
                os.append(str(a['os']))
                active.append(str(a['active']))
                url.append(str(a['url']))
                valid_countries.append(str(a['valid_countries']))
                top_countries.append(str(a['top_countries']))
                release_date.append(str(a['release_date'])[0:10])
                updated_date.append(str(a['updated_date'])[0:10])
                in_app_purchases.append(str(a['in_app_purchases']))
                rating.append(str(a['rating']))
                price.append(str(a['price']))
                rating_count.append(str(a['rating_count']))
                version.append(str(a['version']))
                humanized_worldwide_last_month_downloads.append(str(a['humanized_worldwide_last_month_downloads']["downloads"]))
                humanized_worldwide_last_month_revenue.append(str(a['humanized_worldwide_last_month_revenue']["revenue"]))
                feature_graphic.append(str(a['feature_graphic']))
                short_description.append(str(a['short_description']))
                screenshot_urls.append(str(a['screenshot_urls']))
            for apptopiaid in app_id:

                params = {
                    'id': apptopiaid,
                    'country_iso': co,
                    'date_from':apptopia_date,
                    'date_to':apptopia_date
                }
                url1 = apptopiaurl + urlencode(params)
                apptopiaURL = parse.quote(url1)
                adtURL = 'http://144.24.31.243/apptopia/call?url='+apptopiaURL #通过这个链接去请求
                responseValue = requests.get(adtURL)
                
                apptopia_estimates = responseValue.json()
                day_downloads.append(apptopia_estimates[0]["downloads"])
                total_revenue.append(apptopia_estimates[0]["total_revenue"])
                iap_revenue.append(apptopia_estimates[0]["iap_revenue"])
                ad_revenue.append(apptopia_estimates[0]["ad_revenue"])
                dau.append(apptopia_estimates[0]["dau"])
                mau.append(apptopia_estimates[0]["mau"])
                download_revenue.append(apptopia_estimates[0]["download_revenue"])
                arpu.append(apptopia_estimates[0]["arpu"])
                
            APPdatafram = {}
            APPdatafram["appid"] = app_id 
            APPdatafram["categories"] = categories 
            APPdatafram["canonical_country"] = canonical_country
            APPdatafram["humanized_name"] = humanized_name
            APPdatafram["publisher_name"] = publisher_name
            APPdatafram["icon_url"] = icon_url
            APPdatafram["os"] = os
            APPdatafram["active"] = active
            APPdatafram["url"] = url
            APPdatafram["valid_countries"] = valid_countries
            APPdatafram["top_countries"] = top_countries
            APPdatafram["release_date"] = release_date
            APPdatafram["updated_date"] = updated_date
            APPdatafram["in_app_purchases"] = in_app_purchases
            APPdatafram["rating"] = rating
            APPdatafram["price"] = price
            APPdatafram["rating_count"] = rating_count
            APPdatafram["version"] = version
            APPdatafram["humanized_worldwide_last_month_downloads"] = humanized_worldwide_last_month_downloads
            APPdatafram["humanized_worldwide_last_month_revenue"] = humanized_worldwide_last_month_revenue
            APPdatafram["feature_graphic"] = feature_graphic
            APPdatafram["short_description"] = short_description
            APPdatafram["screenshot_urls"] = screenshot_urls

            
            APPdatafram = pd.DataFrame.from_dict(APPdatafram)
            df3 = pd.merge(APPIDdatafram,APPdatafram,how='inner',on='appid')
            print(df3)
            
            #################进入apptopia爬取阶段
            
            

            # mysql = with_mysql()
            tt=df3.to_json(orient ='records')
            table = "sensortower_categoryRankings"
            savedata(table,tt)
            print("入库成功")

BRalltopselling_free
{'category': 'all', 'chart_type': 'topselling_free', 'country': 'BR', 'date': 20241013, 'auth_token': 'ST0_zxixGoMC_JvC24JFqtLFzxs'}
{'error': 'Invalid authentication token.'}


KeyError: 'ranking'